# Fitting your Aggregation to the Duration Curve of Your Data

Suppose a model has to decide whether a region needs a peak power plant, one that only runs
when the load exceeds 550 MW. The model doesn't care *when* those hours occur, only *how
many* there are. That is what a **duration curve** shows: an attribute's values sorted from
highest to lowest, so that you can read off how much of the time each level is exceeded.

In this tutorial we reduce two weeks of solar irradiance and load to two typical days, first
with the cluster mean and then with `Distribution()`, and count the hours above 550 MW each
time. By the end you will have

* read a duration curve,
* seen the mean flatten it and `Distribution()` keep it,
* rebuilt the values of `Distribution()` by hand, and
* measured what keeping the duration curve costs.

This is part 1 of five [duration-curve tutorials](../index.md#duration-curves).

## 1  Two weeks of hourly data

We take **1–14 April 2010** from the example data: hourly solar irradiance (`GHI`) and
electricity `Load`. Two typical days for two weeks is a deliberately tight budget; it makes
the differences easy to see.

Every run in this tutorial uses the same settings, so we write them down once in a small
function. Only its `representation` argument will change. `preserve_column_means=False`
switches off tsam's rescaling step, so that we see the values exactly as the representation
produced them.

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

import tsam
from tsam import ClusterConfig, Distribution
from tsam.plot import compare_series

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("../../data/testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-04-01":"2010-04-14", ["GHI", "Load"]]
UNITS = {"GHI": "W/m²", "Load": "MW"}
COLORS = {"original": "#000000", "mean": "#0072B2", "distribution": "#009E73"}


def aggregate(representation):
    return tsam.aggregate(
        data,
        n_clusters=2,
        period_duration="1D",
        cluster=ClusterConfig(method="hierarchical", representation=representation),
        preserve_column_means=False,
    )

Sort each attribute from highest to lowest and plot it against the share of time:

In [ ]:
compare_series(
    {"original": data},
    mode="duration_curve",
    units=UNITS,
    colors=COLORS,
    title="Duration curves of the two weeks",
).show()

Read the load curve from the left. The highest hour reaches 562.5 MW, and after about 3% of
the time the curve has dropped below 550 MW. Count those hours:

In [ ]:
int((data["Load"] > 550).sum())

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert int((data["Load"] > 550).sum()) == 11

11 of the 336 hours. That is the number our peak plant depends on.

## 2  Reduce with the mean

The `mean` representation averages a typical day's days hour by hour.
`result.reconstructed` lays the typical days back out over the two weeks, one copy for every
day each of them represents. Its duration curve is what a model that works with the typical
days sees.

In [ ]:
mean = aggregate("mean")

compare_series(
    {"original": data, "mean": mean.reconstructed},
    mode="duration_curve",
    reference="original",
    units=UNITS,
    colors=COLORS,
    title="The mean flattens the duration curves",
).show()

The blue curves are flatter than the original: their highest values are lower, and the lowest
load is higher. The highest load is now 544 MW, so it never reaches 550 MW. Count again:

In [ ]:
int((mean.reconstructed["Load"] > 550).sum())

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert int((mean.reconstructed["Load"] > 550).sum()) == 0

None. A model fed with these typical days would see no need for the peak plant. Averaging
days hour by hour smooths their highs and lows away.

## 3  Fit the duration curve instead

Change only the representation:

In [ ]:
distribution = aggregate(Distribution())

compare_series(
    {
        "original": data,
        "mean": mean.reconstructed,
        "distribution": distribution.reconstructed,
    },
    mode="duration_curve",
    reference="original",
    units=UNITS,
    colors=COLORS,
    title="Distribution() keeps the duration curves",
).show()

The green curves follow the original closely, at the top as well. Count the hours above
550 MW once more:

In [ ]:
int((distribution.reconstructed["Load"] > 550).sum())

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert int((distribution.reconstructed["Load"] > 550).sum()) == 9

9 hours, close to the original's 11. The model would now see that the peak plant has work
to do.

## 4  Build one Representative by hand

Now we build the larger of the two typical days by hand from the orginal data. It represents 9
days, which contribute 9 × 24 = 216 hourly GHI values. Sort them, cut them into 24 groups of
9, and average each group:

In [ ]:
# Querry the time series for all days that are represented by the larger typical day.

## How many days each typical day represents: here {0: 9.0, 1: 5.0}.
days_per_typical_day = distribution.cluster_counts

## The larger typical day is the one that represents the most days.
larger_typical_day = max(days_per_typical_day, key=days_per_typical_day.get)

## Mark the days it represents, then take their positions (0 = 1 April).
is_represented = distribution.cluster_assignments == larger_typical_day
represented_days = np.flatnonzero(is_represented)

## Cut out the 24 hourly GHI values of each of these days.
daily_ghi = [data["GHI"].iloc[day * 24 : (day + 1) * 24] for day in represented_days]

## Pool them into one series: 9 days of 24 hours = 216 values.
pooled_ghi = pd.concat(daily_ghi)

# Sort, group and average the data
## Sort the 216 values from lowest to highest.
sorted_ghi = np.sort(pooled_ghi)

## Cut the sorted values into 24 groups of 9, one row per group.
ghi_groups = sorted_ghi.reshape(24, 9)

## Average each group: 24 values, one per hour of the typical day.
ghi_by_hand = ghi_groups.mean(axis=1)

Put them next to the values of the typical day from `Distribution()`, sorted in the same way.
These are the five highest:

In [ ]:
# The larger typical day as Distribution() built it, hour by hour.
distribution_day = distribution.cluster_representatives.loc[larger_typical_day]

# Its GHI values, still in hour order.
distribution_day_ghi = distribution_day["GHI"]

# Sort them from lowest to highest, like the values built by hand.
ghi_from_distribution = np.sort(distribution_day_ghi)

# Side by side; tail() shows the five highest.
comparison_table = pd.DataFrame(
    {"by hand": ghi_by_hand, "Distribution()": ghi_from_distribution}
)
comparison_table.tail().round(1)

They agree. Check all 24:

In [ ]:
np.allclose(ghi_by_hand, ghi_from_distribution)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert np.allclose(ghi_by_hand, ghi_from_distribution)

Each value of the typical day is the average of 9 hours that lie next to each other on the
duration curve. That's why the fitted curve follows the original so closely. It's also why
its very top, the average of the 9 brightest hours (628 W/m²), stays below the brightest hour
itself (668 W/m²).

## 5  What the typical day looks like

So far we've only looked at sorted values. Now look at the typical day itself, hour by hour,
in front of the 9 days it represents (grey):

In [ ]:
members = {
    data.index[day * 24].strftime("%d %b"): data.iloc[
        day * 24 : (day + 1) * 24
    ].reset_index(drop=True)
    for day in represented_days
}
compare_series(
    {
        "mean": mean.cluster_representatives.loc[larger_typical_day],
        "distribution": distribution.cluster_representatives.loc[larger_typical_day],
    },
    background=members,
    background_label="original days",
    units=UNITS,
    colors=COLORS,
    title="The larger typical day, hour by hour",
).show()

Both typical days rise and fall at the same hours. `Distribution()` places its highest value
where the mean is highest, its second highest where the mean is second highest, and so on.
But its middays are brighter and its load peak is higher: it keeps the values that the mean
averages away. Placing the values in time is a choice of its own;
[part 4](04_ordering.ipynb) compares the five ways tsam can make it.

## 6  What it costs

tsam measures both sides of this trade-off. `rmse_duration` compares the sorted values,
`rmse` compares hour by hour. Both are in normalized units, and lower is better:

In [ ]:
pd.DataFrame(
    {
        name: {
            "rmse_duration": result.accuracy.weighted_rmse_duration,
            "rmse": result.accuracy.weighted_rmse,
        }
        for name, result in {"mean": mean, "distribution": distribution}.items()
    }
).T.round(3)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert (
    distribution.accuracy.weighted_rmse_duration
    < mean.accuracy.weighted_rmse_duration / 3
)
assert distribution.accuracy.weighted_rmse > mean.accuracy.weighted_rmse

`Distribution()` cuts the duration-curve error to less than a third, while its hour-by-hour
error rises a little. Its values are more extreme than the mean's, so at a given hour they
tend to sit further from what actually happened on each day.

## What you've learned

* A duration curve sorts an attribute's values, so it shows for how many hours each level is
  exceeded: here, 11 hours above 550 MW.
* The `mean` representation flattens the duration curve (no hour above 550 MW), and
  `Distribution()` keeps it (9 hours).
* `Distribution()` sorts all hours of a typical day's days, cuts them into 24 equal groups and
  averages each group. You rebuilt this by hand.
* It trades a little hour-by-hour accuracy for a much better duration curve.

Where to go next:

* [Part 2: Local or global](02_scope.ipynb) — fit each typical day's own duration curve, or
  the whole series'.
* [Part 3: Keeping the extremes](03_extremes.ipynb) — keep the real highest and lowest values.
* [Part 4: Which values occur together](04_ordering.ipynb) — place the fitted values in time.
* [Part 5: Making peaks coincide on purpose](05_coincident_peaks.ipynb) — when pairing the
  highest values with each other is exactly what you want.
* [Representation](../../explanation/how-aggregation-works/03_representation.ipynb) — why the
  representations calculate their values the way they do.